# SAME latent space

SAME is the autoencoder Stable Audio 3 generates into, so it is the space a prior would live in.
Two questions:

1. **Round-trip** — what does `decode(encode(x))` cost us?
2. **Transport** — is MP3 damage a constant offset in that space? If so, `decode(encode(x) + v)`
   is the cheapest restoration imaginable, and the floor everything else must beat.

Run `scripts/run_experiments.py` first. Audio here is level-matched to −14 LUFS with one common
headroom gain, so nothing clips and no comparison is decided by loudness.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display

sys.path.insert(0, "../src")
from grooveback import audio as ga

ART = Path("../artifacts/same")
RESULTS = json.loads((ART / "results.json").read_text())
BAND = "16000-20000"

ORDER = [
    "1_input_mp3",
    "2_output_decode_encode_plus_shift",
    "ref_decode_encode_input",
    "ref_clean_master",
    "ref_decode_encode_clean",
]


def play(directory):
    """Play a listening set, filename first so it is clear what produced it."""
    def rank(path):
        return ORDER.index(path.stem) if path.stem in ORDER else 99

    for path in sorted(Path(directory).glob("*.wav"), key=rank):
        print(path.stem)
        display(Audio(str(path)))


def spectra(items, title, floor=-110):
    """Overlay 1/12-octave smoothed spectra of several files."""
    fig, ax = plt.subplots(figsize=(11, 4))
    for path, label in items:
        x, sr = ga.load(path)
        mono = x.mean(0)
        spec = np.abs(np.fft.rfft(mono * np.hanning(len(mono))))
        freqs = np.fft.rfftfreq(len(mono), 1 / sr)
        edges = np.geomspace(20, sr / 2, 120)
        idx = np.digitize(freqs, edges)
        binned = np.array([
            spec[idx == i].mean() if (idx == i).any() else np.nan
            for i in range(1, len(edges))
        ])
        ax.semilogx(edges[:-1], 20 * np.log10(binned / len(mono) + 1e-12), label=label)
    ax.set(xlim=(20, 22050), ylim=(floor, None), xlabel="Hz", ylabel="dB", title=title)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    plt.show()

## The material

12 s excerpts at 1:00 and 3:00 of each source — one fixed rule, intro and main body.

| name | what it is |
|---|---|
| `aerofunk_*` | clean master |
| `an2_*` | real 128 kbps YouTube rip |
| `an2_vinyl_*` | vinyl transfer of the same track, aligned to the rip — the one real reference |
| `codec` | 6 s clean asset, content stops near 5 kHz; the severe case |

The vinyl runs 0.0125% fast, so it slides about 45 ms against the rip across the track.
`grooveback.align` fits speed and offset together; afterwards the two agree to 12 samples.

## 1. Round-trip

The residual is `input − output`, saved before any level matching, so it is the honest difference.

In [ ]:
bands = list(next(iter(RESULTS["roundtrip"].values()))["band_delta_db"])
print(f"{'excerpt':26s}{'resid dB':>9s}" + "".join(f"{b.split('-')[0]:>8s}" for b in bands))
for key in sorted(RESULTS["roundtrip"]):
    row = RESULTS["roundtrip"][key]
    print(
        f"{key:26s}{row['residual_below_signal_db']:9.1f}"
        + "".join(f"{row['band_delta_db'][b]:8.2f}" for b in bands)
    )

In [ ]:
rt = ART / "roundtrip"
spectra(
    [
        (rt / "an2_60s/1_input.wav", "input (real 128k rip)"),
        (rt / "an2_60s/2_output_decode_encode_same_s.wav", "same-s"),
        (rt / "an2_60s/2_output_decode_encode_same_l.wav", "same-l"),
    ],
    "Real rip: the decoder invents a top end that was never there",
)

spectra(
    [
        (rt / "an2_vinyl_60s/1_input.wav", "input (vinyl transfer)"),
        (rt / "an2_vinyl_60s/2_output_decode_encode_same_s.wav", "same-s"),
        (rt / "an2_vinyl_60s/2_output_decode_encode_same_l.wav", "same-l"),
    ],
    "Same music with real top end: the round-trip barely moves it",
)

**The decoder hallucinates, and the vinyl proves it.** The rip is dead above 16 kHz and comes back
with +22.6 dB (same-s) or +11.9 dB (same-l) of content there. The vinyl transfer of the *same
music*, which actually has top end, moves by about 1 dB. The decoder only fabricates where the
input is empty.

So `decode(encode(degraded))` is not a no-op, and any latent-space method has to be judged against
*it*, never against the input file — otherwise the autoencoder's behaviour gets credited to the
method.

In [ ]:
play(ART / "roundtrip/an2_60s")

## 2. Transport vector

Take a clean master, run it through a LAME encode and back, and encode both to latents. The mean of
`z_clean − z_degraded` over 32 windows is a single 256-dim vector `v`.

The whole method is `decode(encode(x) + v)`. One addition, no model, no iteration. Fitted on
Aerofunk only, with every excerpt below held out.

In [ ]:
print(f"{'':16s}{'|v|':>7s}{'variance expl.':>16s}{'direction agree':>17s}{'worst':>8s}")
for key, block in sorted(RESULTS["transport"].items()):
    s = block["shift"]
    print(
        f"{key:16s}{s['norm']:7.2f}{s['variance_explained']:15.0%}"
        f"{s['cosine_mean']:17.2f}{s['cosine_min']:8.2f}"
    )

Windows agree strongly on the **direction** of the damage, but a constant explains well under half
of its **magnitude**. Two things worth noticing:

- The 192k vector points the same way at roughly half the length in both spaces — what you would
  want if this tracked bitrate rather than fitting noise.
- **SAME-L's space is the less linear of the two.** The better autoencoder explains less of the
  damage with a constant, and its worst window agrees far less. Reconstruction quality and "damage
  is a simple direction here" are separate properties.

### What the shift does

`ceiling` is `decode(encode(clean))` — where a *faithful* reconstruction lands, given the
autoencoder is not transparent. It is a calibration point, **not** an upper bound: the encoder is
not the decoder's inverse, and band energy is not a distance. Overshooting it means more energy
than accuracy would put there, which counts against the shift rather than for it.

In [ ]:
for key, block in sorted(RESULTS["transport"].items()):
    print(f"\n{key}   (dB in {BAND} Hz)")
    cols = f"  {'excerpt':16s}{'input':>9s}{'no shift':>10s}{'+ shift':>10s}{'ceiling':>10s}"
    print(cols + "   scored against")
    for name, rows in block["excerpts"].items():
        ceiling = rows.get("ref_decode_encode_clean", {}).get(BAND)
        ceiling_col = "         —" if ceiling is None else f"{ceiling:10.2f}"
        print(
            f"  {name:16s}{rows['1_input_mp3'][BAND]:9.2f}"
            f"{rows['ref_decode_encode_input'][BAND]:10.2f}"
            f"{rows['2_output_decode_encode_plus_shift'][BAND]:10.2f}"
            f"{ceiling_col}   {rows['scored_against']}"
        )

Two problems are visible, both of which the synthetic-only version of this experiment hid.

**On the real rip the autoencoder does nearly all the work, and the shift overshoots.** `an2_60s`
at 128k goes from −21.9 to −3.1 on the round-trip *alone*, then the shift pushes it to +1.5, past
the −0.2 ceiling. That is the hallucination from part 1 doing the lifting, with the vector adding
energy on top.

**At 192 kbps the autoencoder destroys more than the codec does.** Aerofunk arrives 2.7 dB down and
comes back 10.5 dB down. No method operating inside this space can help there — it starts behind.

In [ ]:
tr = ART / "transport"
spectra(
    [
        (tr / "an2_60s_128k_same-s/1_input_mp3.wav", "1 input (real 128k rip)"),
        (tr / "an2_60s_128k_same-s/2_output_decode_encode_plus_shift.wav", "2 output: + v"),
        (tr / "an2_60s_128k_same-s/ref_decode_encode_input.wav", "ref: no shift"),
        (tr / "an2_60s_128k_same-s/ref_clean_master.wav", "ref: aligned vinyl"),
        (tr / "an2_60s_128k_same-s/ref_decode_encode_clean.wav", "ref: ceiling"),
    ],
    "an2_60s @ 128k, same-s — against its own vinyl transfer",
)

In [ ]:
play(ART / "transport/an2_60s_128k_same-s")

In [ ]:
play(ART / "transport/aerofunk_60s_128k_same-s")

## What to listen for

Every number above is energy, and energy is not perception. Specifically:

- Does `2_output` sound **better** than `1_input`, or just brighter?
- Does it beat `ref_decode_encode_input` — is the shift doing anything the autoencoder was not
  already doing on its own? On the real rip the numbers say mostly no.
- On AN-2, does it move toward `ref_clean_master`, the actual vinyl, or past it?

Whatever a prior does later has to beat this. One 256-float addition is the floor.